<a href="https://colab.research.google.com/github/arturfurlan07-debug/Mini-Projeto/blob/main/LoL_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://wiki.leagueoflegends.com/en-us/"
HEADERS = {"User-Agent": "LoLWikiScraper-Fatec/1.0 (projeto academico"}

campeoes = [
    "Aatrox", "Ahri", "Akali", "Annie", "Ashe", "Caitlyn", "Darius",
    "Ekko", "Ezreal", "Garen", "Jinx", "Lee Sin", "Leona", "Lux",
    "Master Yi", "Miss Fortune", "Sett", "Teemo", "Vi", "Yasuo", "Zed",
]

In [ ]:
def buscar_html(url):
    try:
        resposta = requests.get(url, headers=HEADERS, timeout=10)
        resposta.raise_for_status()
        return resposta.text

    except requests.exceptions.Timeout:
        print("Erro: o site demorou demais para responder (timeout).")
    except requests.exceptions.ConnectionError:
        print("Erro: não foi possível conectar à internet ou ao site.")
    except requests.exceptions.HTTPError as erro:
        print(f"Erro: o site respondeu com um erro HTTP ({erro}).")
    except requests.exceptions.RequestException as erro:
        print(f"Erro inesperado ao tentar acessar o site: {erro}")
    return None


def extrair_entre(linhas, marcador_inicial, marcador_final):
    try:
        posicao_inicial = linhas.index(marcador_inicial)
        posicao_final = linhas.index(marcador_final, posicao_inicial)
    except ValueError:
        return ""
    trecho = linhas[posicao_inicial + 1:posicao_final]
    return ", ".join(texto for texto in trecho if texto not in {",", "|"})


def pegar_campeao(nome):
    html = buscar_html(URL + nome.replace(" ", "_"))
    if html is None:
        return None
    pagina = BeautifulSoup(html, "html.parser")
    linhas = pagina.get_text("\n", strip=True).split("\n")
    posicao_tags = linhas.index("Tag(s):") if "Tag(s):" in linhas else 0
    linhas_infobox = linhas[posicao_tags:]
    return {
        "nome": nome,
        "titulo": linhas[posicao_tags - 1] if posicao_tags else "",
        "rota": extrair_entre(linhas_infobox, "Role(s):", "Store price:"),
        "classe": extrair_entre(linhas_infobox, "Class", "Legacy class"),
        "tags": extrair_entre(linhas_infobox, "Tag(s):", "Role(s):"),
        "lancamento": extrair_entre(linhas_infobox, "Release date", "Last changed"),
    }

In [ ]:
for nome in campeoes:
    dados = pegar_campeao(nome)
    if dados:
        print(" | ".join(dados.values()))
    else:
        print(f"{nome}: sem dados")

Aatrox | The Darkin Blade | Top | Juggernaut | Sustained Damage, Self Healing, Physical | 2013-06-13
Ahri | The Nine-Tailed Fox | Middle | Burst | Mobile, Burst, Magic | 2011-12-14
Akali | The Rogue Assassin | Top, Middle | Assassin | Mobile, Burst, Magic | 2010-05-11
Annie | The Dark Child | Middle | Burst | Summon, Burst, Magic | 2009-02-21
Ashe | The Frost Archer | Bottom, Support | Marksman | Auto-attack, Crowd Control, Physical | 2009-02-21
Caitlyn | The Sheriff of Piltover | Bottom | Marksman | Auto-attack, Burst, Physical | 2011-01-04
Darius | The Hand of Noxus | Top | Juggernaut | Duelist, Self Healing, Physical | 2012-05-23
Ekko | The Boy Who Shattered Time | Jungle, Middle | Assassin | Burst, Mobile, Magic | 2015-05-29
Ezreal | The Prodigal Explorer | Bottom | Marksman | Long Range, Mobile, Physical | 2010-03-16
Garen | The Might of Demacia | Top | Juggernaut | Durable, Sustained Damage, Physical | 2010-04-27
Jinx | The Loose Cannon | Bottom | Marksman | Auto-attack, Hypercar